In [ ]:
#https://gitlab.com/tomaye/abstrct

import os
import re
import pandas as pd
from google.colab import drive

# --- 1. Mount Google Drive ---
# This will open a permission window to access your files.
drive.mount('/content/drive')

# --- 2. Define Paths ---
# Path to the main project folder in your Drive
main_folder = '/content/drive/My Drive/ProjectFolder/'  # <--- Generic folder name

# Names of the subfolders containing the data
data_subfolders = ['folder1', 'folder2', 'folder3']  # <--- Generic subfolder names

def create_argument_dataset_colab(base_dir, subfolders):
    """
    Traverses subdirectories to create an argument dataset.

    Args:
        base_dir (str): Path to the main folder (e.g., '/content/drive/My Drive/ProjectFolder/').
        subfolders (list): A list with the names of the subfolders to process.

    Returns:
        pd.DataFrame: A pandas DataFrame with the combined data.
    """
    complete_dataset = []

    # Iterate over each specified subfolder
    for folder_name in subfolders:
        current_dir = os.path.join(base_dir, folder_name)

        if not os.path.exists(current_dir):
            print(f"Warning: Folder '{current_dir}' not found")
            continue

        print(f"Processing: '{folder_name}'...")

        # Build a list with the base names of the files (e.g., '1565457')
        base_files = set()
        for file_name in os.listdir(current_dir):
            base_files.add(os.path.splitext(file_name)[0])

        for base in sorted(list(base_files)):
            txt_file = os.path.join(current_dir, base + ".txt")
            ann_file = os.path.join(current_dir, base + ".ann")

            if os.path.exists(txt_file) and os.path.exists(ann_file):
                with open(ann_file, 'r', encoding='utf-8') as f:
                    annotations = f.readlines()

                mapped_texts = {}
                for line in annotations:
                    if line.startswith('T'):
                        parts = line.strip().split('\t')
                        text_id = parts[0]
                        text = parts[2]
                        mapped_texts[text_id] = text

                for line in annotations:
                    if line.startswith('R'):
                        parts = line.strip().split('\t')
                        relation_info = parts[1].split()

                        raw_classification = relation_info[0]
                        argument_id = relation_info[1].split(':')[1]
                        topic_id = relation_info[2].split(':')[1]

                        topic = mapped_texts.get(topic_id)
                        argument = mapped_texts.get(argument_id)

                        if topic and argument:
                            if 'Support' in raw_classification:
                                classification = 'SUPPORT'
                            elif 'Attack' in raw_classification:
                                classification = 'ATTACK'
                            else:
                                classification = 'UNDEFINED'

                            # Add the record to the list, including the new column 'source'
                            complete_dataset.append({
                                'subject': folder_name,
                                'topic': topic,
                                'argument': argument,
                                'ground_truth_stance': classification
                            })

    # Convert the list of dictionaries into a pandas DataFrame
    return pd.DataFrame(complete_dataset)

# --- 3. Run the Function and Create the Dataset ---
df_dataset = create_argument_dataset_colab(main_folder, data_subfolders)

In [ ]:
df_dataset.head()

In [ ]:
# --- 4. Exibir e Salvar o Resultado ---
if not df_dataset.empty:
    print("\n--- Dataset criado com sucesso! ---")

    # Mostra as 5 primeiras e as 5 últimas linhas do dataset
    display(df_dataset.head())
    display(df_dataset.tail())

    print(f"\nTotal de registros no dataset: {len(df_dataset)}")

    # **ALTERAÇÃO AQUI**: Salvar o dataset na pasta raíz do Google Drive
    pasta_raiz_drive = '/content/drive/My Drive/...'
    caminho_saida_csv = os.path.join(pasta_raiz_drive, 'AbstRT.csv')

    df_dataset.to_csv(caminho_saida_csv, index=False, encoding='utf-8')

    print(f"\nDataset saved in: '{caminho_saida_csv}'")
else:
    print("\nNo data found.")